# Sommelier — chạy toàn bộ pipeline trên Kaggle 2×T4

Chạy hết luồng: nhạc → diarization → tách chồng tiếng → ASR → refinement → export,
rồi **nghiệm thu từng thứ đã đổi** thay vì tin là nó chạy đúng.

## Ba môi trường, không phải một

Pipeline chạy DiariZen và Qwen3-ASR ở **tiến trình riêng** vì phụ thuộc của chúng
đánh nhau với env chính:

| env | ở đâu | vì sao tách |
|---|---|---|
| `sommelier_env` | `/kaggle/temp` | pipeline chính: WhisperX, pyannote, audio-separator, onnxruntime |
| `diarizen_env` | `/kaggle/temp` | cần `numpy 1.26.4` + `scipy 1.13.1` và bản pyannote-audio fork của DiariZen |
| `qwen3_env` | `/kaggle/temp` | cần `transformers>=5.13` + `huggingface-hub>=1.5`, WhisperX thì cần `hub 0.33` |

Cả ba env **và** cache của `uv` phải nằm trên **cùng một filesystem**. `uv` hardlink
từ cache sang venv khi cùng ổ, nên ba env dùng chung một bản torch (~3.5G tổng); khác ổ
hoặc đặt `UV_LINK_MODE=copy` thì mỗi env giữ một bản riêng (~10.5G) và Kaggle hết đĩa
giữa chừng. Ô 2 tự chọn ổ còn nhiều chỗ nhất và dừng ngay nếu dưới 22G.

Mọi lệnh cài đều đi kèm `-c constraints.txt` ghim `torch==2.8.0`. Không có nó,
`accelerate==1.12.0` trong env DiariZen kéo về **torch 2.14 và cả bộ CUDA 13 thứ hai**.

Pipeline tìm chúng qua `DIARIZEN_PYTHON` / `QWEN3_PYTHON`.

## Cái đã đổi mà notebook này kiểm

| | Trước | Giờ |
|---|---|---|
| Tách chồng tiếng | Sidon (mù) + USEF | **chỉ USEF**, điều kiện hoá bằng enrollment |
| Cửa sổ mixture | 6–12s ghép từ solo A + solo B | **chính overlap, nới tới 2s** |
| BS-RoFormer | `ep_317`, autocast | **`ep_368`**, native fp16, `chunk_duration` |
| Ngưỡng hát | `0.35` — cao hơn trần model, chưa từng kích hoạt | **`0.12`** |
| Ngưỡng độ dài khi CẮT | `0.30` — không lọc gì | **`0.96`** (3 block 320ms) |
| Truy vết | không có | **`orig_spans` / `crosses_cut` / `gap_before`** |
| Nhiễu phi âm nhạc | không nhìn thấy | **`noise_score`** trên mỗi segment |

> Các ngưỡng suy ra từ một lần dump 527 nhãn PANNs của **ba** bản ghi, chỉ một trong ba
> có tiếng hát thật. Chạy xong hãy nghe, đừng tin ngay.

## 1. Máy chạy được gì

In [ ]:
import os, shutil
os.environ["MPLBACKEND"] = "Agg"

# KHONG dat UV_LINK_MODE=copy: uv mac dinh HARDLINK tu cache sang venv khi ca hai
# nam cung mot filesystem, va do la thu duy nhat lam ba env torch vua o Kaggle.
# Copy thi moi env giu mot ban torch rieng: ~3.5GB x 3 thay vi ~3.5GB tong.
os.environ.pop("UV_LINK_MODE", None)

!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!free -g | head -2
print()
for path in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    if os.path.exists(path):
        u = shutil.disk_usage(path)
        print(f"{path:18} tong {u.total/2**30:6.1f}G  trong {u.free/2**30:6.1f}G")


## 2. Thư mục và code

Ô kiểm tra nhánh ở dưới **dừng hẳn** nếu clone ra commit chưa có các thay đổi —
vì clone nhầm thì mọi thứ vẫn chạy trơn tru, chỉ là chạy pipeline cũ, và kết quả
trông y như thật.

In [ ]:
import os, shutil

BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()

# Cache va ca ba env phai nam CUNG MOT filesystem, neu khong uv khong hardlink
# duoc va moi env phai chep rieng ~3.5GB torch. Chon o con nhieu cho nhat.
cands = [p for p in ("/kaggle/temp", "/tmp", BASE_DIR) if os.path.exists(p)]
ENV_ROOT = max(cands, key=lambda p: shutil.disk_usage(p).free)
free = shutil.disk_usage(ENV_ROOT).free / 2**30
print(f"env + cache -> {ENV_ROOT}  ({free:.1f}G trong)")
if free < 22:
    raise SystemExit(
        f"Chi con {free:.1f}G tren {ENV_ROOT}. Ba env torch can ~20G ke ca khi hardlink.\n"
        "Restart session (Run > Factory reset) roi chay lai truoc khi cai gi.")

os.environ["UV_CACHE_DIR"] = os.path.join(ENV_ROOT, ".uv_cache")

PROJECT_DIR  = os.path.join(BASE_DIR, 'sommerlier')
PIPELINE_DIR = os.path.join(PROJECT_DIR, 'podcast-pipeline')
AUDIO_DIR    = os.path.join(BASE_DIR, 'vi_audio')
OUT_DIR      = os.path.join(BASE_DIR, 'out')

ENV_DIR          = os.path.join(ENV_ROOT, 'sommelier_env')
DIARIZEN_ENV_DIR = os.path.join(ENV_ROOT, 'diarizen_env')
QWEN3_ENV_DIR    = os.path.join(ENV_ROOT, 'qwen3_env')

BRANCH = 'tgss'          # <-- nhanh DA PUSH

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.chdir(BASE_DIR)
!git clone --depth 1 --branch {BRANCH} https://github.com/foresst123/sommerlier.git
assert os.path.exists(PIPELINE_DIR), "clone hong"
print("commit:", os.popen(f"git -C {PROJECT_DIR} log -1 --oneline").read().strip())


In [ ]:
must_exist = {
    "utils/mixture_window.py": "cua so mixture moi cho USEF",
    "utils/provenance.py":     "truy vet orig_spans / gap_before",
    "utils/noise_map.py":      "diem nhieu phi am nhac",
}
missing = [f"  - {p}  ({why})" for p, why in must_exist.items()
           if not os.path.exists(os.path.join(PIPELINE_DIR, p))]
must_be_gone = [p for p in ("sidon_worker.py", "utils/spectral_restore.py")
                if os.path.exists(os.path.join(PIPELINE_DIR, p))]
if missing or must_be_gone:
    raise SystemExit(
        "Nhanh nay CHUA co cac thay doi.\n"
        + ("Thieu:\n" + "\n".join(missing) + "\n" if missing else "")
        + ("Con sot (le ra da xoa): " + ", ".join(must_be_gone) + "\n" if must_be_gone else "")
        + f"\nCommit va push nhanh '{BRANCH}' roi chay lai o nay.")
print("Nhanh dung: co mixture_window / provenance / noise_map, Sidon da go.")

## 3a. Env chính (~10–14 phút)

`requirements.txt` của repo, bỏ đúng **một** dòng: `git+DiariZen` — nó kéo theo bản
pyannote fork làm hỏng pyannote 4.0.7 mà WhisperX cần. DiariZen sống ở env riêng bên dưới.

`nemo-toolkit[asr]` **đã được gỡ khỏi `requirements.txt`**: nó chỉ được import bởi
`models/sortformer.py`, mà dòng import module đó trong `model_loader.py` đang bị comment —
không ai nạp. Nó còn làm file không giải được: nemo 2.2.0 ghim `numba==0.61.0` chống lại
`numba==0.61.2` ở đây, nên `uv pip install -r` hỏng ngay trước khi cài gì.

Nếu ô này báo `No solution found`, nhánh bạn clone chưa có bản sửa đó.

In [ ]:
!pip install -q uv

# Chan moi lan resolve nang torch len ban khac. Khong co no, `accelerate` trong
# env DiariZen keo ve torch 2.14 + ca bo CUDA 13 thu hai va lam day dia.
CONSTRAINTS = os.path.join(BASE_DIR, 'constraints.txt')
open(CONSTRAINTS, 'w').write("torch==2.8.0\ntorchaudio==2.8.0\ntorchvision==0.23.0\n")

TORCH = "torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0"
CU126 = "--extra-index-url https://download.pytorch.org/whl/cu126"

def disk(tag):
    u = shutil.disk_usage(ENV_ROOT)
    print(f"  [{tag}] con {u.free/2**30:.1f}G tren {ENV_ROOT}")

!uv venv --allow-existing --python 3.12 {ENV_DIR}

req_src = os.path.join(PIPELINE_DIR, 'requirements.txt')
lines = [l for l in open(req_src).read().splitlines() if 'DiariZen.git' not in l]
req_main = os.path.join(BASE_DIR, 'requirements_main.txt')
open(req_main, 'w').write("\n".join(lines))

!uv pip install --python {ENV_DIR} {TORCH} {CU126}
!uv pip install --python {ENV_DIR} -r {req_main} -c {CONSTRAINTS} {CU126} --index-strategy unsafe-best-match
disk("sau env chinh")


## 3b. Env DiariZen

Ba chỗ dễ vỡ, nên có bước ép version cuối cùng: pyannote hay tự nâng numpy/scipy
lên bản làm hỏng numba, và DiariZen cần **bản pyannote-audio fork của chính nó**
chứ không phải bản trên PyPI.

In [ ]:
DZ_PY  = f"{DIARIZEN_ENV_DIR}/bin/python"
DZ_REV = "844f5555b0a98acd0931511fc641a8c5b8ba92c7"
DZ_URL = "https://github.com/BUTSpeechFIT/DiariZen.git"

!rm -rf {DIARIZEN_ENV_DIR}
!uv venv {DIARIZEN_ENV_DIR} --python 3.12
!uv pip install --python {DZ_PY} {TORCH} --index-url https://download.pytorch.org/whl/cu126
disk("torch diarizen")

!uv pip install --python {DZ_PY} -c {CONSTRAINTS} numpy==1.26.4 scipy==1.13.1 pandas==2.2.3 numba==0.59.1 llvmlite==0.42.0
!uv pip install --python {DZ_PY} -c {CONSTRAINTS} soundfile librosa==0.10.2.post1 matplotlib==3.9.4 pyparsing accelerate==1.12.0
!uv pip install --python {DZ_PY} -c {CONSTRAINTS} "git+{DZ_URL}@{DZ_REV}#subdirectory=pyannote-audio"
!uv pip install --python {DZ_PY} -c {CONSTRAINTS} git+{DZ_URL}@{DZ_REV}
!uv pip install --python {DZ_PY} -c {CONSTRAINTS} speechbrain==1.0.2 toml==0.10.2 wrapt==2.3.0 psutil==7.0.0
# Ep lai: pyannote hay tu nang numpy/scipy len ban lam hong numba.
!uv pip install --python {DZ_PY} numpy==1.26.4 scipy==1.13.1 --no-deps
disk("sau env diarizen")

!{DZ_PY} -c "import numpy,scipy,torch;print('numpy',numpy.__version__,'scipy',scipy.__version__,'torch',torch.__version__,'cuda',torch.cuda.is_available())"
!{DZ_PY} -c "from diarizen.pipelines.inference import DiariZenPipeline; print('DiariZenPipeline OK')"


## 3c. Env Qwen3-ASR

Tách vì `transformers>=5.13` cần `huggingface-hub>=1.5`, còn WhisperX ở env chính
ghim `hub 0.33`. Hai cái không sống chung được.

In [ ]:
Q_PY = f"{QWEN3_ENV_DIR}/bin/python"
!uv venv --allow-existing {QWEN3_ENV_DIR} --python 3.12
!uv pip install --python {Q_PY} {TORCH} {CU126}
!uv pip install --python {Q_PY} -c {CONSTRAINTS} "transformers>=5.13.0" "huggingface-hub>=1.5.0" accelerate soundfile librosa
disk("sau env qwen3")
!{Q_PY} -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"


## 3d. Dọn cache và kiểm env chính

In [ ]:
# Xoa cache CHI SAU KHI ca ba env da cai xong: xoa som se pha cac hardlink
# chua tao va buoc env sau tai lai tu dau.
!uv cache clean
!rm -rf {os.environ["UV_CACHE_DIR"]}
disk("sau khi don cache")

import glob
# pkg_resources cua setuptools moi lam pyannote/nemo no khi import; thay bang no-op.
for site in glob.glob(os.path.join(ENV_DIR, 'lib', 'python*', 'site-packages')):
    open(os.path.join(site, 'pkg_resources.py'), 'w').write('def declare_namespace(name): pass\n')
    print("da va pkg_resources trong", site)

python_bin = os.path.join(ENV_DIR, 'bin', 'python')
check = """
import torch, librosa, soundfile, pandas, pydub, onnxruntime
import pyannote.audio, whisperx, faster_whisper
from panns_inference import SoundEventDetection
from audio_separator.separator import Separator
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.device_count())
print("onnxruntime:", onnxruntime.get_available_providers())
"""
probe = os.path.join(BASE_DIR, 'probe.py'); open(probe, 'w').write(check)
import subprocess
r = subprocess.run([python_bin, probe], capture_output=True, text=True)
print(r.stdout or r.stderr[-3000:])
assert "torch" in r.stdout, "env chinh thieu thu vien"
if "CUDAExecutionProvider" not in r.stdout:
    print("\nCANH BAO: onnxruntime khong thay CUDA -> USEF chay CPU, cham nhung van dung.")


## 4. Trỏ pipeline vào các env worker

`resolve_worker_python` đọc `DIARIZEN_PYTHON` / `QWEN3_PYTHON` trước, rồi mới tới
`worker_envs` trong config, rồi tới đường dẫn quy ước. Đặt env var là chắc nhất.

In [ ]:
os.environ["DIARIZEN_PYTHON"] = DZ_PY
os.environ["QWEN3_PYTHON"]    = Q_PY
for k in ("DIARIZEN_PYTHON", "QWEN3_PYTHON"):
    p = os.environ[k]
    assert os.path.exists(p), f"{k} tro vao {p} — khong ton tai"
    print(f"{k:18} {p}")

## 5. Token HuggingFace — bắt buộc cho bản đầy đủ

In [ ]:
hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for key in ("HUGGINGFACE_TOKEN", "HF_TOKEN"):
        try:
            hf_token = s.get_secret(key) or ""
            if hf_token: break
        except Exception:
            pass
except Exception:
    pass
if not hf_token:
    raise SystemExit("Chua co token. Add-ons > Secrets > them HUGGINGFACE_TOKEN roi chay lai.")
os.environ["HUGGINGFACE_TOKEN"] = hf_token
os.environ["HF_TOKEN"] = hf_token
print("Da lay token tu Kaggle Secrets.")

## 6. Audio đầu vào

In [ ]:
import glob, subprocess
os.makedirs(AUDIO_DIR, exist_ok=True)
EXTS = ('.mp3','.wav','.flac','.m4a','.aac','.ogg','.opus')
found = []
if os.path.exists('/kaggle/input'):
    for ext in EXTS:
        found += glob.glob(f'/kaggle/input/**/*{ext}', recursive=True)
for f in found:
    dst = os.path.join(AUDIO_DIR, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)

audio_files = sorted(p for p in glob.glob(os.path.join(AUDIO_DIR, '*'))
                     if p.lower().endswith(EXTS))
assert audio_files, "Khong tim thay audio trong /kaggle/input — Add Input mot dataset."

total = 0.0
for p in audio_files:
    try:
        d = float(subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                                  "-of","default=nk=1:nw=1",p],
                                 capture_output=True, text=True).stdout.strip() or 0)
    except Exception:
        d = 0.0
    total += d
    print(f"  {os.path.basename(p):44} {d/60:6.1f} phut  {os.path.getsize(p)/1e6:6.1f} MB")
print(f"\n{len(audio_files)} file, tong {total/60:.1f} phut ({total/3600:.2f} gio)")
print("Do 2026-08-22 tren 2xT4: 10 phut audio ~ 22 phut GPU, cong ~3.8 phut nap model moi file.")
print(f"Uoc tinh lan chay nay: ~{total/60*2.2/60:.1f} gio.")

## 7. Bật đủ stage

`config.json` trong repo đang **tắt** diarization/separation/asr/export (commit `c33ce45`,
để chạy riêng bước nhạc). Ô này bật lại.

`music_removal_fallback` để tắt: bước bóc nhạc lần hai trên từng segment, đã thừa vì
nhạc được bóc ở mức sóng ngay đầu luồng. Trên hai bản ghi thật nó chưa từng chạy.

In [ ]:
import json, pathlib
cfg_path = os.path.join(PIPELINE_DIR, 'config.json')
cfg = json.loads(pathlib.Path(cfg_path).read_text(encoding='utf-8'))
profile = cfg['environments']['kaggle']
profile['steps'].update({
    "music_analysis": True, "music_removal": True, "cut_singing": True,
    "music_removal_fallback": False,
    "diarization": True, "separation": True, "asr": True,
    "captioning": False, "refinement": True, "export": True,
})
profile.setdefault('worker_envs', {}).update({"diarizen": DZ_PY, "qwen3": Q_PY})
pathlib.Path(cfg_path).write_text(json.dumps(cfg, indent=4, ensure_ascii=False) + "\n",
                                  encoding='utf-8')
print("steps :", json.dumps(profile['steps']))
print("tse   :", json.dumps(profile['models']['tse']))
print("bsrof :", json.dumps(profile['models']['bs_roformer']))
assert profile['models']['tse']['separator'] == 'usef'
assert 'ep_368' in profile['models']['bs_roformer']['model']

### Ngưỡng — để nguyên lần đầu

| biến | mặc định | căn cứ |
|---|---|---|
| `MUSIC_MAP_THRESHOLD` | `0.10` | phân bố lưỡng cực: chỉ 0.1–3.2% frame nằm giữa 0.02 và 0.35 |
| `MUSIC_MAP_SINGING` | `0.12` | điểm tách sạch đầu tiên: 0 dương tính giả trên 2 file không hát, giữ 23s trên file có hát |
| `MUSIC_MAP_SINGING_MARGIN` | `0.15` | **chưa hiệu chỉnh** — giờ là ràng buộc chặn thật sự |
| `MUSIC_MAP_MIN_SPAN` | `0.32` | đúng một block quyết định của model |
| `MUSIC_MAP_MIN_SPAN_CUT` | `0.96` | ba block; loại 6 span vụn mà chỉ mất 2.9s |
| `NOISE_NOTICEABLE` | `0.10` | trần thực tế ba nhóm nhiễu là 0.28 / 0.26 / 0.07 |

In [ ]:
tuning = {
    # "MUSIC_MAP_SINGING":        "0.12",
    # "MUSIC_MAP_SINGING_MARGIN": "0.15",
    # "MUSIC_MAP_MIN_SPAN_CUT":   "0.96",
    # "NOISE_NOTICEABLE":         "0.10",
}
os.environ.update(tuning)
print(tuning or "dung nguyen mac dinh")

## 8. Xoá checkpoint

**Bắt buộc nếu từng chạy rồi.** `music_map` đọc từ checkpoint nếu có, nên ngưỡng hát
và ngưỡng độ dài mới sẽ không áp lên file đã chạy — span cũ vẫn nguyên và bạn sẽ tưởng
thay đổi không có tác dụng.

In [ ]:
FRESH_RUN = True
if FRESH_RUN:
    for d in (OUT_DIR, os.path.join(PIPELINE_DIR, 'cache')):
        shutil.rmtree(d, ignore_errors=True)
    ledger = os.path.join(AUDIO_DIR, '_sommelier_progress.json')
    if os.path.exists(ledger):
        os.remove(ledger)
    print("da xoa cache, output va so ghi tien do")
else:
    print("GIU checkpoint — nguong moi se KHONG ap len file da chay")

## 9. Chạy

`--keep_models` giữ model trong VRAM giữa các stage: nhanh hơn khi nhiều file, nhưng
**đỉnh VRAM cao hơn**. Bỏ cờ này nếu gặp CUDA OOM.

`--max_hours` gom file thành từng lượt; `--gpu_1`/`--gpu_2` chia DiariZen và Qwen3 sang hai card.

In [ ]:
import sys
site = os.path.join(ENV_DIR, 'lib', 'python3.12', 'site-packages')
os.environ["LD_LIBRARY_PATH"] = (os.environ.get("LD_LIBRARY_PATH", "")
                                 + f":{site}/nvidia/cudnn/lib:{site}/torch/lib")
os.chdir(PIPELINE_DIR)
!CUDA_VISIBLE_DEVICES=0,1 {python_bin} main.py \
    --audio_dir "{AUDIO_DIR}" \
    --save_path "{OUT_DIR}" \
    --env kaggle \
    --lang vi \
    --gpu_1 0 --gpu_2 1 \
    --max_hours 5 \
    --tse --panns --vad --ASRMoE --llm_refinement \
    --keep_models

---

# Nghiệm thu

Từ đây không chạy model nào — chỉ đọc file kết quả và hỏi xem từng thay đổi có thật
sự xảy ra không. Ô nào `assert` hỏng là chỗ đó chưa chạy.

## 10. Bước nhạc

In [ ]:
import json, glob
runs = sorted(glob.glob(os.path.join(OUT_DIR, '*', '01_music', 'music_map.json')))
assert runs, f"khong thay 01_music/music_map.json trong {OUT_DIR}"
maps = {}
for path in runs:
    name = os.path.basename(os.path.dirname(os.path.dirname(path)))
    m = json.load(open(path)); maps[name] = m
    spans = m.get('spans', [])
    kinds = {}
    for s in spans:
        kinds[s['kind']] = kinds.get(s['kind'], 0) + 1
    print(f"=== {name}: {len(spans)} span {kinds}")
    print(f"    nhac duoi loi {m.get('music_seconds',0):.1f}s | hat {m.get('singing_seconds',0):.1f}s"
          f" | nhac khong loi {m.get('song_seconds',0):.1f}s")
    print(f"    cat {m.get('removed_seconds',0):.1f}s, con {m.get('kept_stretches',0)} manh")
    short = [s for s in spans if s['kind'] in ('song','singing') and s['end']-s['start'] < 0.96]
    print(f"    span cat ngan hon 0.96s: {len(short)}   (phai la 0)")

    mus = [s for s in spans if s['kind'] == 'music']
    ex  = [s for s in spans if s['kind'] in ('song','singing')]
    ov  = [(a,b) for a in mus for b in ex if a['start'] < b['end'] and b['start'] < a['end']]
    wasted = sum(min(a['end'],b['end']) - max(a['start'],b['start']) for a,b in ov)
    print(f"    span music chong len vung bi cat: {len(ov)}/{len(mus)}  ({wasted:.1f}s boc nhac roi vut)")
    print("      ^ no ky thuat da biet: PAD_SECONDS cong vao TUNG span doc lap")

## 11. Diarization và tách chồng tiếng

In [ ]:
from collections import Counter
finals = [f for f in sorted(glob.glob(os.path.join(OUT_DIR, '*', '*.json')))
          if 'intermediate' not in f]
docs = {}
for f in finals:
    d = json.load(open(f))
    if not isinstance(d, dict) or 'segments' not in d:
        continue
    docs[os.path.splitext(os.path.basename(f))[0]] = d
assert docs, "khong thay transcript — buoc export chua chay"

for name, d in docs.items():
    segs = d['segments']
    spk = Counter(s['speaker'] for s in segs)
    tse = sum(1 for s in segs if s.get('tse'))
    print(f"=== {name}: {len(segs)} segment, {len(spk)} speaker, TSE {tse} ({tse/len(segs)*100:.1f}%)")
    for s, c in spk.most_common():
        t = sum(1 for x in segs if x['speaker'] == s and x.get('tse'))
        print(f"      {s:14} {c:4d} seg   qua TSE: {t:3d}")
    durs = sorted(x['end'] - x['start'] for x in segs); n = len(durs)
    at_cap = sum(1 for x in durs if x >= 19.5)
    print(f"    do dai: p50 {durs[n//2]:.1f}s  max {durs[-1]:.1f}s | dồn sát trần 20s: {at_cap}")
    print("    (tren 2 ban ghi cu: TSE cham 4.8% va 1.0% — thap la binh thuong)")

## 12. Truy vết

`gap_before = None` nghĩa là khoảng nghỉ trước segment đó **chứa một vết cắt**, nên
không dùng được để học nhịp hội thoại. Đây là thứ ngăn corpus học "đối đáp nhanh
0.05 giây" từ chỗ vốn cách nhau 10.8 giây nhạc.

In [ ]:
for name, d in docs.items():
    segs = d['segments']; meta = d.get('metadata', {})
    print(f"=== {name}")
    assert 'orig_spans' in segs[0], "khong co truy vet — clone commit cu hoac export tat"
    glued  = sum(1 for s in segs if s.get('crosses_cut'))
    broken = sum(1 for s in segs[1:] if s.get('gap_before') is None)
    gaps   = [s['gap_before'] for s in segs[1:] if s.get('gap_before') is not None]
    print(f"    metadata.provenance: {meta.get('provenance')}")
    print(f"    segment dan tu 2 manh: {glued}")
    print(f"    khoang nghi bi vet cat pha: {broken}   <-- bo qua khi hoc timing")
    if gaps:
        g = sorted(gaps)
        print(f"    khoang nghi dung duoc: {len(g)} | p50 {g[len(g)//2]:.2f}s"
              f" | ngat loi (am): {sum(1 for x in gaps if x < 0)}")
    bad = [s['index'] for s in segs
           if abs(sum(o['end']-o['start'] for o in (s.get('orig_spans') or []))
                  - (s['end']-s['start'])) > 0.05]
    print(f"    orig_spans khong khop do dai: {len(bad)}   (phai la 0)")
    assert not bad, f"truy vet sai: {bad[:5]}"

## 13. Điểm nhiễu phi âm nhạc

In [ ]:
thr = float(os.environ.get("NOISE_NOTICEABLE", "0.10"))
for name, d in docs.items():
    scores = sorted(s['noise_score'] for s in d['segments'] if s.get('noise_score') is not None)
    if not scores:
        print(f"{name}: chua do nhiem — music_analysis co chay khong?"); continue
    n = len(scores); over = sum(1 for x in scores if x >= thr)
    print(f"=== {name}: {n} segment co diem")
    print(f"    p50 {scores[n//2]:.4f} | p90 {scores[int(n*0.9)]:.4f} | max {scores[-1]:.4f}")
    print(f"    vuot {thr}: {over} segment ({over/n*100:.1f}%)")
    if scores[-1] > 0.30:
        print("    ^ cao hon tran 0.28 do duoc truoc day — ban ghi nay ban hon bo hieu chinh")

## 14. ASR

In [ ]:
for name, d in docs.items():
    segs = d['segments']
    empty = sum(1 for s in segs if not (s.get('text') or '').strip())
    print(f"=== {name}: {len(segs)} segment, {empty} khong co chu ({empty/len(segs)*100:.1f}%)")
    for s in segs[:3]:
        print(f"  [{s['index']}] {s['start']:7.2f}-{s['end']:7.2f} {s['speaker']} gap={s.get('gap_before')}")
        print(f"      ensemble : {(s.get('text') or '')[:96]}")
        for k in ('text_whisper', 'text_phowhisper', 'text_qwen3'):
            v = (s.get(k) or '').strip()
            if v: print(f"      {k:15}: {v[:96]}")
        print()

## 15. Đóng gói để tải về

In [ ]:
import numpy as np, soundfile as sf, librosa
CLIP_PAD, MAX_CLIPS = 4.0, 10
PACK = os.path.join(BASE_DIR, 'sommelier_review')
shutil.rmtree(PACK, ignore_errors=True)

def to_cut(kept, t):
    if not kept: return t
    for a, b, c in kept:
        if a <= t < b: return c + (t - a)
    return None

for name, m in maps.items():
    src = next((p for p in audio_files
                if os.path.splitext(os.path.basename(p))[0] == name), None)
    after = os.path.join(OUT_DIR, name, '01_music', 'after_music.flac')
    if not src or not os.path.exists(after):
        print(f"{name}: thieu file goc hoac after_music.flac"); continue
    y0, sr0 = librosa.load(src, sr=16000, mono=True)
    y1, _   = librosa.load(after, sr=16000, mono=True)
    kept = (m.get('timeline') or {}).get('kept', [])
    d = os.path.join(PACK, name, 'clips'); os.makedirs(d, exist_ok=True)
    for kind in ('music', 'song', 'singing'):
        for i, s in enumerate([x for x in m['spans'] if x['kind'] == kind][:MAX_CLIPS], 1):
            c = (s['start'] + s['end']) / 2
            a0, b0 = max(0, int((c-CLIP_PAD)*sr0)), min(len(y0), int((c+CLIP_PAD)*sr0))
            sf.write(os.path.join(d, f"{kind}_{i:02d}_{s['start']:.1f}s_before.wav"), y0[a0:b0], sr0)
            cc = to_cut(kept, c)
            if cc is None: cc = to_cut(kept, s['start']) or 0.0
            a1, b1 = max(0, int((cc-CLIP_PAD)*sr0)), min(len(y1), int((cc+CLIP_PAD)*sr0))
            if b1 > a1:
                sf.write(os.path.join(d, f"{kind}_{i:02d}_{s['start']:.1f}s_after.wav"), y1[a1:b1], sr0)
    print(f"{name}: {len(os.listdir(d))} clip")

shutil.make_archive(os.path.join(BASE_DIR, 'sommelier_review'), 'zip', PACK)
shutil.make_archive(os.path.join(BASE_DIR, 'sommelier_out'), 'zip', OUT_DIR)
!ls -lh {BASE_DIR}/*.zip
print("\n-> panel Output ben phai")

In [ ]:
from IPython.display import Audio, display
name = list(maps)[0]
for b in sorted(glob.glob(os.path.join(PACK, name, 'clips', '*_before.wav')))[:4]:
    a = b.replace('_before.wav', '_after.wav')
    print(os.path.basename(b).replace('_before.wav', ''))
    print("  truoc:"); display(Audio(b))
    if os.path.exists(a):
        print("  sau:"); display(Audio(a))
    else:
        print("  sau: (doan da bi cat, khong co moi noi)")

## Nếu có gì đó không đúng

| Triệu chứng | Nguyên nhân |
|---|---|
| Ô 2b dừng, báo thiếu file | Nhánh chưa push. `git push origin tgss` rồi chạy lại |
| `No solution found ... numba` khi cài | Nhánh cũ còn `nemo-toolkit` trong `requirements.txt`. Bản sửa đã gỡ nó — pull lại |
| `DiariZenPipeline OK` không in ra | Env DiariZen hỏng — thường do numpy bị pyannote nâng lên. Chạy lại ô 3b |
| `worker interpreter not found` | `DIARIZEN_PYTHON`/`QWEN3_PYTHON` sai. Chạy lại ô 4 |
| `No space left on device` | Env hoặc cache nằm khác ổ nhau → `uv` phải chép thay vì hardlink. Ô 2 phải in cùng một đường dẫn cho cả ba env. Đừng đặt `UV_LINK_MODE=copy` |
| Log nhắc `torch 2.14` / `cuda-toolkit 13` | Thiếu `-c constraints.txt`. Đó là torch thứ hai, gấp đôi dung lượng |
| Hết chỗ dù đã hardlink | Restart session (Run > Factory reset). Kaggle không trả lại chỗ khi chỉ xoá thư mục |
| CUDA OOM | Bỏ `--keep_models`. Nếu vẫn OOM, hạ `models.diarizen.batch_size` |
| `music_map.json` có 0 span | Tìm dòng `Music map:` trong log; không có thì `steps.music_analysis` đang false |
| `singing_seconds` vẫn bằng 0 | Có thể **đúng** — trên 3 bản ghi cũ đều bằng 0 vì hát luôn rơi vào `song` (không ai nói đè). Xem cột `song` trước khi nghi ngờ |
| Span cắt ngắn hơn 0.96s | Checkpoint cũ. `FRESH_RUN = True` |
| `orig_spans` không có trong JSON | Clone commit cũ, hoặc `export` tắt |
| Mọi `noise_score` là `None` | `music_analysis` tắt — điểm nhiễu đến từ cùng lần quét PANNs |
| TSE chạm gần 0% | Bình thường nếu ít chồng tiếng (2 bản ghi cũ: 4.8% và 1.0%) |
| `no_enroll` nhiều trong log | Speaker không đủ 1.5s audio sạch — diarization vỡ vụn hoặc nhạc phủ kín |

## Chưa được kiểm chứng

- **USEF chưa từng chạy thật** trong đợt sửa này; cửa sổ mixture mới chỉ qua test với backend giả.
- **Bộ chặn người-thứ-ba đã gỡ** cùng Sidon. Nhiều job được nhận hơn trước; USEF về lý thuyết nén giọng thứ ba nhưng chưa ai đo.
- **`qc_sim_threshold = 0.2`** — comment trong code ghi rõ "NOT calibrated".
- **Điểm nhiễu chưa gặp bản ghi ngoài trời**; ngưỡng 0.10 đặt từ ba file trong nhà.
- **`SINGING_MARGIN = 0.15`** giờ là ràng buộc chặn thật sự, chưa từng hiệu chỉnh.